In [1]:
# ============================================================
# 🥷 AIBO v7.4.0 ワンプッシュ起動セル
# Phase 1: HF cache · Phase 2: 依存 · Phase 3: 起動 · Phase 4: 完成 · Phase 5: API 公開
# ============================================================
import os, sys, importlib, subprocess, shutil, time

# Phase 0: import warm-up は無効化 (VRAM OOM の原因となるため)
# torch/nunchaku を background thread で import すると CUDA コンテキスト初期化が
# 競合し、main thread のメモリプール確保が失敗するケースを確認。
# 4 秒の最適化を捨てて VRAM 安定性を優先。

# ─── Phase 1: Drive mount + HF cache ─────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=False)
# 🆕 CASE-A (RECON-002 第2段 · 案A): モデルは HF Hub から /content(NVMe)へ直 DL する。
# Drive FUSE 越しの巨大 safetensors 読み込み(=truncation→外国人化)を構造的に発生させない。
# Drive はコード(AIBO_ROOT)の置き場としてのみ使い、モデルの一次ソースからは外す。
GDRIVE_HF_CACHE = "/content/drive/MyDrive/aibo_hf_cache"   # sync_back_to_drive() 用に定義のみ保持
CONTENT_HF_CACHE = "/content/aibo_hf_cache"
# 🛡️ FIX-1 (IMPL-003 / H1 対策): /content/aibo_hf_cache が「Drive を指す stale symlink」だと
#   直 DL が Drive に書かれ、ロードで FUSE mmap → OSError errno 19(ENODEV)で落ちる。
#   symlink なら必ず除去してから実ディレクトリを作り、Drive 非経由をアサートする(silent fail 禁止)。
if os.path.islink(CONTENT_HF_CACHE):
    os.unlink(CONTENT_HF_CACHE)          # 旧 notebook が作った Drive 直結 symlink を除去
elif os.path.isdir(CONTENT_HF_CACHE) and os.path.realpath(CONTENT_HF_CACHE).startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] HF cache が Drive 配下: {os.path.realpath(CONTENT_HF_CACHE)}")
os.makedirs(CONTENT_HF_CACHE, exist_ok=True)
# ★ ガード: 用意した cache が Drive を指していないことを起動時にアサート
_real_cache = os.path.realpath(CONTENT_HF_CACHE)
if _real_cache.startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] FATAL: HF cache が Drive 上 ({_real_cache})。NVMe 直 DL に失敗。")
print(f"✅ [CASE-A] HF cache = {_real_cache} (NVMe・Drive 非経由を確認)")

# 🗑️ CASE-A で撤去した事故経路:
#   - tar pipe(CACHE_SYNC_TIMEOUT=180)による Drive→/content 一括コピー
#   - timeout 時の shutil.rmtree(CONTENT_HF_CACHE)+os.symlink(GDRIVE→CONTENT)(=Drive 直結)
#   これらが「180s で部分コピー→直結 symlink→FUSE 越し巨大読み込み→truncation」の確定的事故経路だった。

# /root/.cache/huggingface → /content/aibo_hf_cache symlink
# (env 変数を無視する library 対策、全 HF DL を NVMe に固定。Drive ではなく NVMe を指す)
ROOT_HF = "/root/.cache/huggingface"
os.makedirs("/root/.cache", exist_ok=True)
if os.path.islink(ROOT_HF) or os.path.exists(ROOT_HF):
    subprocess.run(["rm", "-rf", ROOT_HF], check=False)
os.symlink(CONTENT_HF_CACHE, ROOT_HF)
print(f"✅ HF cache symlink: {ROOT_HF} → {CONTENT_HF_CACHE}")

# env 変数も /content (NVMe) を指す
for k in ["HF_HOME", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_HUB_CACHE"]:
    os.environ[k] = CONTENT_HF_CACHE
# 🆕 CASE-A: HF_HUB_ENABLE_HF_TRANSFER は deprecated 警告が出るため設定しない。
#   高速 DL は hf_xet(既定・huggingface_hub>=0.32)に委ねる。
#   hf_xet の稀なサイズ一致破損は C0 検証ゲート(IMPL-001)が from_pretrained 直前に捕捉する前提。
print(f"✅ HF cache → {CONTENT_HF_CACHE} (NVMe)")

# ─── Phase 1.4: HF token 注入 (userdata 経由・コードに直書きしない) ───
# gated 2 repo(FLUX.1-dev / FLUX.1-Redux-dev)の DL に HF token が必須(G0 調査)。
try:
    from google.colab import userdata
    _hf_tok = userdata.get('HF_TOKEN')   # 既存の NGROK_AUTH_TOKEN と同じ Colab Secrets 仕組み
except Exception as _e:
    _hf_tok = None
    print(f"⚠️ [CASE-A] userdata.get('HF_TOKEN') 取得失敗: {type(_e).__name__}: {_e}")
if _hf_tok:
    os.environ['HF_TOKEN'] = _hf_tok        # 値はログに出さない
    try:
        from huggingface_hub import login
        login(token=_hf_tok)
        print("✅ [CASE-A] HF login OK (token は非表示)")
    except Exception as _e:
        print(f"⚠️ [CASE-A] HF login 失敗: {type(_e).__name__}: {_e}")
else:
    # 握りつぶさない: token 不在を明示警告(gated DL は 401/403 で失敗する)
    print("⚠️ [CASE-A] HF_TOKEN 未設定。gated モデル(FLUX.1-dev/Redux)の DL は失敗します")

# ─── Phase 1.5: HF Hub → /content 直 DL (snapshot_download 一次化) ───
# default cache-dir 方式(local_dir 指定しない=巨大単一ファイルの resume 堅牢性のため · RECON-002 F3)。
# 出口は C0 検証ゲート(IMPL-001)が各 from_pretrained 直前に守る。
from huggingface_hub import snapshot_download
# (repo_id, allow_patterns, ignore_patterns) — None は無指定(allow=None で repo 丸ごと / ignore=None で除外なし)。
_HF_REPOS = [
    # gated (HF_TOKEN 必須)
    ("black-forest-labs/FLUX.1-dev", None, ["flux1-dev.safetensors", "transformer/*"]),  # IMPL-009: bf16 transformer 2塊を除外(Nunchaku INT4 使用)
    ("black-forest-labs/FLUX.1-Redux-dev", None, None),
    # public (token 不要)
    # 🛡️ FIX-2 (IMPL-003 / H2 対策): nunchaku は int4 をピン。A100(Ampere)= int4(get_precision 準拠)。
    #   fp4 は Blackwell 専用なので落とさない。無駄 DL(fp4 ~6.5GB)削減 + DL/ロードの precision 整合。
    ("nunchaku-tech/nunchaku-flux.1-dev",          # INT4 transformer 本体(307→nunchaku-ai に自動追従)
     ["svdq-int4_r32-flux.1-dev.safetensors", "*.json", "README*"], None),
    ("mit-han-lab/svdq-int4-flux.1-fill-dev", None, None),       # Fill transformer
    ("Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0", None, None),
    ("XLabs-AI/flux-ip-adapter", None, None),
    ("guozinan/PuLID", None, None),
    ("ByteDance/Hyper-SD", ["Hyper-FLUX.1-dev-8steps-lora.safetensors"], None),  # IMPL-009: 8steps LoRA 単一化
]
print(f"\n📥 [CASE-A] HF Hub → /content 直 DL 開始 ({len(_HF_REPOS)} repo)")
_t_dl = time.time()
for _repo, _allow, _ignore in _HF_REPOS:
    try:
        snapshot_download(repo_id=_repo, allow_patterns=_allow, ignore_patterns=_ignore)   # allow/ignore=None は無指定。既ロード分は resume/skip。実体は /content(NVMe)
        print(f"  ✅ [CASE-A] OK: {_repo}")
    except Exception as _e:
        # 握りつぶさない: gated は token 無/未同意で 401/403。明示 fail-fast(縮退に逆戻りさせない)。
        print(f"  ❌ [CASE-A] FAIL: {_repo} :: {type(_e).__name__}: {_e}")
        raise
print(f"✅ [CASE-A] 直 DL 完了: {time.time()-_t_dl:.1f}s")

# ─── Phase 2: torchsde 事前 install ─────────────────────────
for dep in ['torchsde']:
    try:
        __import__(dep)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep], check=True)

# ─── Phase 3: AIBO 起動 ─────────────────────────────────────
print("\n" + "=" * 60)
print("🚀 AIBO v7.4.0 起動シーケンス")
print("=" * 60)

AIBO_ROOT = "/content/drive/MyDrive/aibo_v7"

# BOM 除去
for fname in ["01_config.py", "02_colab_setup.py", "03_identity_engine.py",
              "04_pipeline_manager.py", "05_orchestrator.py", "06_ui.py",
              "07_main.py", "08_face_refiner.py"]:
    fpath = os.path.join(AIBO_ROOT, fname)
    with open(fpath, "rb") as f:
        c = f.read()
    if c.startswith(b'\xef\xbb\xbf'):
        with open(fpath, "wb") as f:
            f.write(c[3:])

if AIBO_ROOT not in sys.path:
    sys.path.insert(0, AIBO_ROOT)

# モジュール import (キャッシュクリア)
for m in list(sys.modules):
    if m.startswith(("01_","02_","03_","04_","05_","06_","07_","08_")):
        del sys.modules[m]

t_imp = time.perf_counter()
mod01 = importlib.import_module("01_config")
mod02 = importlib.import_module("02_colab_setup")  # numpy 自動チェック + 必要時のみ自動再起動
mod03 = importlib.import_module("03_identity_engine")
mod04 = importlib.import_module("04_pipeline_manager")
mod05 = importlib.import_module("05_orchestrator")
mod06 = importlib.import_module("06_ui")
mod07 = importlib.import_module("07_main")
mod08 = importlib.import_module("08_face_refiner")
print(f"⏱️ import: {time.perf_counter() - t_imp:.2f}s")

SystemConfig = mod01.SystemConfig
GenerationConfig = mod01.GenerationConfig
IdentityConfig = mod01.IdentityConfig
StudioMode = mod01.StudioMode
AiboMain = mod07.AiboMain

# AiboMain 起動
t_boot = time.perf_counter()
aibo = AiboMain()
if hasattr(aibo, "run"):
    aibo.run(enable_gradio=False)  # A方式運用 · Phase G スキップ
print(f"⏱️ AiboMain 起動: {time.perf_counter() - t_boot:.2f}s")

orchestrator = aibo.orchestrator
pm = orchestrator.pm
ie = orchestrator.ie
ipa = ie.ip_adapter
gen_cfg = GenerationConfig()
id_cfg = IdentityConfig()

# ─── Phase 4: Phase 1 完成状態セットアップ ──────────────────
print("\n" + "=" * 60)
print("🔧 Phase 1 完成状態セットアップ")
print("=" * 60)
tf = pm._shared_transformer
if pm.pipe_cnet is None:
    pm.ensure_controlnet()
pm._cn_forward_wrapped = False
pm._wrap_transformer_forward_for_cn()
if pm.pipe_cnet.image_encoder is None:
    pm.pipe_cnet.image_encoder = ipa._image_encoder
    pm.pipe_cnet.feature_extractor = ipa._feature_extractor
ipa.set_scale(pm.pipe_base, id_cfg.ip_adapter_weight)
ipa.set_scale(pm.pipe_cnet, id_cfg.ip_adapter_weight)
print(f"✅ forward={tf.forward.__qualname__}")
print(f"✅ IP-Adapter scale={id_cfg.ip_adapter_weight}")

print("\n" + "=" * 60)
print("🎉 v7.4.0 Phase 1 完成 · 即生成可能!")
print("=" * 60)
import sys as _sys
_sys.stdout.flush()

# ─── Phase 5: FastAPI + ngrok 公開 ───────────────────────────
import traceback as _tb
_sys.stdout.flush()
print("\n" + "=" * 60, flush=True)
print("🚀 FastAPI + ngrok 公開", flush=True)
print("=" * 60, flush=True)

try:
    import threading, requests
    for pkg in ["fastapi", "pyngrok"]:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    from pyngrok import ngrok, conf
    from google.colab import userdata

    srv = importlib.import_module("09_fastapi_server")

    # 既起動チェック
    already_running = False
    try:
        r = requests.get("http://localhost:8000/api/system/status", timeout=2)
        if r.status_code == 200 and r.json().get("orchestrator_attached"):
            already_running = True
            print("ℹ️ FastAPI 既起動、スキップ", flush=True)
    except: pass

    if not already_running:
        srv.attach_orchestrator(orchestrator, pm)
        threading.Thread(
            target=lambda: srv.run_server(host="0.0.0.0", port=8000, log_level="warning"),
            daemon=True, name="aibo-fastapi"
        ).start()
        time.sleep(3)

    r = requests.get("http://localhost:8000/api/system/status", timeout=5)
    b = r.json()
    print(f"✅ FastAPI · GPU={b['gpu_name']} · VRAM={b['vram_used_gb']:.1f}/{b['vram_total_gb']:.1f} GB", flush=True)

    # ngrok
    conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
        ngrok.kill()
        time.sleep(1)
    except: pass

    public_url = ngrok.connect(8000, "http").public_url
    time.sleep(3)

    # ngrok 疎通確認 (リトライ付き)
    for i in range(5):
        try:
            r = requests.get(f"{public_url}/api/system/status",
                             headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
            if r.status_code == 200:
                break
        except: pass
        time.sleep(2)

    env_content = f"NEXT_PUBLIC_API_URL={public_url}\nNEXT_PUBLIC_NGROK_SKIP_WARNING=true"
    with open(f"{AIBO_ROOT}/.env.local.latest.txt", "w") as f:
        f.write(env_content)

    print(f"\n🔌 API URL: {public_url}", flush=True)
    print("\n" + "=" * 60, flush=True)
    print("📋 PC の .env.local にコピー", flush=True)
    print("=" * 60, flush=True)
    print(env_content, flush=True)
    print("\n" + "=" * 60, flush=True)
    print("🌐 ブラウザで開く (PC で npm run dev が走ってる前提)", flush=True)
    print("=" * 60, flush=True)
    print("http://localhost:3000", flush=True)
    print("\n🎉 ワンプッシュ起動完了 🥷", flush=True)
except Exception as _e:
    print(f"\n❌ Phase 5 で例外発生: {type(_e).__name__}: {_e}", flush=True)
    _tb.print_exc()
    raise

def sync_back_to_drive():
    """/content cache の内容を Drive に書き戻し (新規 DL の永続化用)"""
    if os.path.islink(CONTENT_HF_CACHE):
        print("ℹ️ symlink モード · Drive 直結のため sync 不要")
        return
    t = time.time()
    try:
        _cmd = f'tar cf - -C "{CONTENT_HF_CACHE}" . | tar xf - -C "{GDRIVE_HF_CACHE}"'
        subprocess.run(["bash", "-c", _cmd], capture_output=True, text=True, timeout=300)
        print(f"✅ Drive 同期完了 (tar): {time.time()-t:.1f}s")
    except subprocess.TimeoutExpired:
        print(f"⚠️ Drive 同期 timeout (300s) · 手動で再試行してください")
    except Exception as _e:
        print(f"⚠️ Drive 同期失敗: {_e}")


✅ [CASE-A] HF cache = /content/aibo_hf_cache (NVMe・Drive 非経由を確認)
✅ HF cache symlink: /root/.cache/huggingface → /content/aibo_hf_cache
✅ HF cache → /content/aibo_hf_cache (NVMe)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ [CASE-A] HF login OK (token は非表示)

📥 [CASE-A] HF Hub → /content 直 DL 開始 (8 repo)


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

  ✅ [CASE-A] OK: black-forest-labs/FLUX.1-dev


...

  ✅ [CASE-A] OK: ByteDance/Hyper-SD
✅ [CASE-A] 直 DL 完了: 2.9s

🚀 AIBO v7.4.0 起動シーケンス


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
ERROR:AIBO_v7:❌ MODULE DESYNC 検出(committed manifest と不一致 → build STOP):
   03_identity_engine.py  runtime=3b297f80  expected=a7556198  src=loaded  (/content/drive/MyDrive/aibo_v7/03_identity_engine.py)
   07_main.py  runtime=9f06cc31  expected=6e7f8a1a  src=loaded  (/content/drive/MyDrive/aibo_v7/07_main.py)
   対処: 該当 .py を G:\マイドライブ\aibo_v7\ に再同期(md5 一致を確認)→ Colab 再起動。
   ※ 『sync した』は証拠ではない。『manifest 一致』が verify/ship ゲートの前提。


⏱️ import: 9.02s
[OBS] module manifest | 01=c7d16e5d 02=7e53e4cc 03=3b297f80 04=045100c0 05=4055f4a5 06=4a3232f3 07=9f06cc31 08=48cd93f6 09=ddd35c2a 10=5358f699 11=11005606 15=922fca01 16=5370cb94 17=36f0de36
❌ MODULE DESYNC 検出(committed manifest と不一致 → build STOP):
   03_identity_engine.py  runtime=3b297f80  expected=a7556198  src=loaded  (/content/drive/MyDrive/aibo_v7/03_identity_engine.py)
   07_main.py  runtime=9f06cc31  expected=6e7f8a1a  src=loaded  (/content/drive/MyDrive/aibo_v7/07_main.py)
   対処: 該当 .py を G:\マイドライブ\aibo_v7\ に再同期(md5 一致を確認)→ Colab 再起動。
   ※ 『sync した』は証拠ではない。『manifest 一致』が verify/ship ゲートの前提。


SystemExit: ❌ MODULE DESYNC 検出(committed manifest と不一致 → build STOP):
   03_identity_engine.py  runtime=3b297f80  expected=a7556198  src=loaded  (/content/drive/MyDrive/aibo_v7/03_identity_engine.py)
   07_main.py  runtime=9f06cc31  expected=6e7f8a1a  src=loaded  (/content/drive/MyDrive/aibo_v7/07_main.py)
   対処: 該当 .py を G:\マイドライブ\aibo_v7\ に再同期(md5 一致を確認)→ Colab 再起動。
   ※ 『sync した』は証拠ではない。『manifest 一致』が verify/ship ゲートの前提。

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
exec(open(".../attach_aibo_log_spigot.py", encoding="utf-8").read())

In [ ]:
!tail -n 80 /content/aibo_gen.log

In [ ]:
import hashlib, os, time
from google.colab import drive

# Drive を再マウント(クラウドから最新を引き直す)
try: drive.flush_and_unmount()
except Exception as e: print("unmount skip:", e)
drive.mount('/content/drive', force_remount=True)

base = "/content/drive/MyDrive/aibo_v7"
expected = {"03_identity_engine.py":"a7556198",
            "07_main.py":"6e7f8a1a",
            "05_orchestrator.py":"4055f4a5"}  # 05 は対照(届いてるはず)

for fn, exp in expected.items():
    p = os.path.join(base, fn)
    if not os.path.exists(p):
        print(f"{fn}: ❌ ファイルが無い"); continue
    raw = open(p, "rb").read()
    norm = hashlib.md5(raw.replace(b"\r\n", b"\n")).hexdigest()[:8]
    mtime = time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
    mark = "✅一致" if norm == exp else "❌まだ古い"
    print(f"{fn}: now={norm} expected={exp} {mark} | 更新={mtime}")